# SE.LLMA: does a published weight edit transfer to Swedish?

## Goal

This notebook is a guided experiment, not a training exercise. We load one small language model, apply a published change to two weight matrices, and ask whether it answers Swedish arithmetic more reliably. We also test English translations and simple alternatives.

**Read the explanation, then press ▶ beside each code cell. Work from top to bottom.** Do not use “Run all” on your first visit: you need to inspect the development pilot before freezing the test settings.

**Recorded run:** see the [submission README](https://github.com/abdkar/sellma-weight-space-swedish-submission) for the verified results. This walkthrough starts with empty outputs and generates evidence for your own run.

| Condition | What changes? | Why include it? |
|---|---|---|
| A | Original weights, thinking on | Main reference |
| B | Published patch, thinking on | Our intervention |
| C | Original weights, thinking off | Simple practical alternative |
| C_pen | Original weights, thinking on, repetition penalty 1.08 | A decoding alternative |
| D | Published patch, thinking off | Compare with C to screen for regressions |
| R1 | Random signs on the patch delta | Test whether edit direction matters |
| S1 | Orthogonal, rank-one random delta | A second limited control |

**Key assumptions:** this is one checkpoint, one decoding seed, original synthetic tasks and two control draws. It does not measure general Swedish proficiency or prove the edit's unique mechanism. A short answer can be wrong; a long answer can be unfinished. We measure these separately.

The main endpoint is a correct final answer **and natural completion** within 8,192 or 16,384 output tokens. Smaller budgets are descriptive points on the saved trajectories. The primary regression contrast is **D minus C** with thinking off.

Protocol: [PROTOCOL.md](https://github.com/abdkar/sellma-weight-space-swedish-submission/blob/main/PROTOCOL.md).

## Roadmap: what you will do, and where to stop

| Phase | Cells | Rough time | Stop and check |
|---|---|---|---|
| 1. Setup | 1–5 | 15–25 min (installs + downloads) | GPU is A100/L4; package version printed is **0.2.0**; all tests pass |
| 2. Inspect data | 6–7 | 10–15 min | You read every Swedish question and set `DATA_REVIEWED = True` |
| 3. Build weights | 8 | 5–10 min | "Verified patched / sign / orthogonal" printed; table shows norm_ratio ≈ 1 |
| 4. Smoke + resume | 9–10 | 5 min | Seven conditions produce text; rerun skips everything |
| 5. Development pilot | 11–12 | 10–25 min | Read completion rates and the projection table |
| 6. Freeze | 13 | 5 min | You wrote the decision **before** any test output |
| 7. Test runs | 14–17 | 30–70 min | One cell per weight variant; do not change settings |
| 8. Analysis + report | 18–23 | 30–40 min | Coverage complete; download the results ZIP |

Times are planning guesses, not guarantees; record what actually happens in `EFFORT_LOG.md`.

## Setup

Choose **Runtime → Change runtime type → A100 GPU**, or **L4 GPU** if A100 is unavailable. Colab can allocate a different device, so the next cell checks it. Use your existing allocation; no subscription purchase is needed. T4 lacks native BF16 support and requires a separately declared FP16 pilot if this exact model/engine supports it.

Keep the notebook open while running. Expect model downloads and package installation to take time. There is no guaranteed runtime. The four-hour assignment target includes preparation already performed; keep an honest effort log.

### 1. Check the machine

**Why:** Confirm the notebook is running on a GPU before downloading weights.

**Do:** Run this cell. If it says no GPU, change the runtime type before continuing.

**Expected:** A GPU name and available memory; no model has run yet.

In [ ]:
import sys, subprocess, time
from pathlib import Path
SESSION_START = time.time()
print('Python:', sys.version.split()[0])
check = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
print(check.stdout or check.stderr)
assert check.returncode == 0, 'Select a GPU runtime before continuing.'

### 2. Bring the project into Colab

**Why:** The notebook needs the Python code and reviewed task file beside it. A ZIP avoids putting a GitHub password or token in a cell.

**Do:** Download the repository ZIP from GitHub (Code → Download ZIP), or use the supplied sellma-colab-bundle.zip. Upload that ZIP when prompted. If you already extracted the repository, set PROJECT to its folder.

**Expected:** A project folder containing sellma_wsp/, data/, configs/ and PROTOCOL.md, and the printed code version **0.2.0**. If you see an older version, set REFRESH_PROJECT = True and rerun this cell with a fresh ZIP: an old copy left in /content is the most common cause of a failing Section 8.

In [ ]:
import zipfile, os, shutil
PROJECT = Path('/content/sellma_project')
REFRESH_PROJECT = False  # True = delete the copy in /content and upload a new ZIP.
if REFRESH_PROJECT and PROJECT.exists():
    shutil.rmtree(PROJECT); shutil.rmtree('/content/sellma_upload', ignore_errors=True)
if not (PROJECT / 'pyproject.toml').exists():
    from google.colab import files
    uploaded = files.upload()
    archives = [Path(name) for name in uploaded if name.endswith('.zip')]
    assert len(archives) == 1, 'Upload exactly one project ZIP.'
    staging = Path('/content/sellma_upload')
    staging.mkdir(exist_ok=True)
    with zipfile.ZipFile(archives[0]) as archive:
        for name in archive.namelist():
            target = (staging / name).resolve()
            assert target.is_relative_to(staging.resolve()), 'Unsafe archive path'
        archive.extractall(staging)
    candidates = list(staging.rglob('pyproject.toml'))
    assert len(candidates) == 1, 'Expected one project in the ZIP.'
    shutil.copytree(candidates[0].parent, PROJECT, dirs_exist_ok=True)
os.chdir(PROJECT)
version = (PROJECT / 'sellma_wsp/__init__.py').read_text().split('__version__ = ')[1].split()[0].strip('"')
print('Project:', PROJECT, '| code version:', version)
assert version >= '0.2.0', 'Old project copy: set REFRESH_PROJECT = True and upload the latest ZIP.'

### 3. Install the experiment environment

**Why:** vLLM handles long, batched generation; MLflow records the runs. Versions here are an initial candidate stack and must pass the smoke test.

**Do:** Run once per fresh runtime. Installation can take several minutes. Stop and inspect any error rather than rerunning later cells.

**Expected:** Successful package installation. Actual installed versions will be recorded with every run.

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'vllm==0.33.0', 'transformers==5.20.2'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[colab,test]'], check=True)
print('Dependencies installed. GPU workers will start in fresh processes.')

### 4. Choose output and backup folders

**Why:** Colab local files can disappear. Keep models on local disk and predictions in durable storage.

**Do:** Leave USE_GOOGLE_DRIVE=True for ordinary use and complete Google's mount prompt. If you choose False, download the results ZIP before disconnecting; local copies alone are not durable.

**Expected:** Local model/run folders and, when enabled, a Drive backup destination. No live SQLite database goes on Drive.

In [ ]:
USE_GOOGLE_DRIVE = True
MODELS = Path('/content/sellma_models')
RUNS = Path('/content/sellma_runs')
BACKUP = None
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BACKUP = Path('/content/drive/MyDrive/SELLMA_experiment_backups')
    BACKUP.mkdir(parents=True, exist_ok=True)
print('Local runs:', RUNS, '\nDurable backup:', BACKUP)

## Steps

We separate **development** from **test**. Development tasks help us find software errors and choose a feasible budget. We freeze these decisions before viewing any test output. Inspecting test questions and their checked answers beforehand is allowed; using the model's test answers to tune settings is not.

### 5. Run the local correctness checks

**Why:** A scoring or split error could invalidate the experiment even if the model runs.

**Do:** Run the tests. They cover answer parsing, family splits, arithmetic answers, random-control geometry and resume conflicts.

**Expected:** All tests pass. These are software checks, not evidence that the patch improves the model.

In [ ]:
subprocess.run([sys.executable, '-m', 'pytest', '-q'], check=True)
from sellma_wsp.common import load_json, read_jsonl, save_json
from sellma_wsp.experiment import config, run, phase_dir, estimate, freeze
from sellma_wsp.analysis import analyze, validate_language_audit, write_report
import pandas as pd

### 6. Inspect the development questions

**Why:** Understand the task, its translations and the exact expected answers.

**Do:** Read the paired questions below. The questions are original synthetic examples. Fix wording only before the final freeze.

**Expected:** 10 development concepts, each in Swedish and English, with one checked integer answer.

In [ ]:
tasks = read_jsonl(PROJECT / 'data/tasks.jsonl')
display(pd.DataFrame(tasks).query("split == 'dev'")[['id','language','prompt','gold']])
print(pd.DataFrame(tasks).groupby(['split','language']).size())

### 7. Inspect test questions and regression probes

**Why:** Check the Swedish wording and answers before the model sees the held-out tasks.

**Do:** Review all displayed rows. Test families are separate from development families; numeric variants within a family are analyzed together. Set DATA_REVIEWED=True only after review.

**Expected:** 40 bilingual test concepts and 16 short regression prompts; no test generation yet.

In [ ]:
with pd.option_context('display.max_rows', 120, 'display.max_colwidth', 160):
    display(pd.DataFrame(tasks).query("split != 'dev'")[['id','split','language','prompt','gold']])
DATA_REVIEWED = False  # Change to True after checking the wording and gold answers.

### 8. Download and verify the model variants

**Why:** Use the exact checkpoint and the exact published patch. A similar model is not an equivalent substitute.

**Do:** Run this cell once. It downloads the pinned Qwen checkpoint (it is stored as an index plus one shard), rebuilds the published edit with the authors' own script, and writes three folders: patched, sign (R1) and orthogonal (S1). Allow about 6 GB of disk. If it fails, the last lines of the log are printed; the usual cause is an old project copy (go back to Section 2 with REFRESH_PROJECT = True).

**Expected:** The log ends with 'Verified patched', 'Verified sign' and 'Verified orthogonal'. In the table, every row has norm_ratio ≈ 1.0; the patched rows have cosine_with_patch = 1.0, the controls ≈ 0. Any hash mismatch stops the experiment by design.

In [ ]:
if (MODELS / 'weights_manifest.json').exists():
    print('Weights already prepared in this runtime; reusing', MODELS)
else:
    result = subprocess.run([sys.executable, '-m', 'sellma_wsp', 'prepare', '--root', str(PROJECT), '--models', str(MODELS)],
                            capture_output=True, text=True)
    print(result.stdout[-4000:])
    if result.returncode != 0:
        print(result.stderr[-6000:])
        raise RuntimeError('Weight preparation failed; read the error above before continuing.')
weights = load_json(MODELS / 'weights_manifest.json')
display(pd.DataFrame([{ 'variant': variant, 'tensor': tensor, **diagnostics }
                     for variant, tensors in weights['variants'].items()
                     for tensor, diagnostics in tensors.items()]))

### 9. Run a short smoke test

**Why:** Check loading, thinking templates, finite outputs, scoring and all seven conditions cheaply.

**Do:** Use vLLM first. If it cannot load the exact model within the setup window, inspect the saved log. A declared Transformers fallback uses the same model for every arm and needs a new smoke test and pilot.

**Expected:** Saved development outputs for all conditions. A 128-token smoke cap is deliberately too short for judging capability. **Speed settings:** BATCH_SIZE = 128 sends a whole condition (up to 96 prompts) to vLLM in one call, so long answers are generated in parallel; ENFORCE_EAGER = False keeps CUDA graphs on. Only if this cell crashes during 'CUDA graph capture' set ENFORCE_EAGER = True (slower) and rerun. These values must stay the same for pilot and test.

In [ ]:
ENGINE = 'vllm'
DTYPE = 'bfloat16'
BATCH_SIZE = 128        # one vLLM call per condition; do not lower to 8 (8k-token runs become ~10x slower)
ENFORCE_EAGER = False   # True only if CUDA-graph capture fails
COMMON = dict(engine=ENGINE, dtype=DTYPE, batch_size=BATCH_SIZE, backup_dir=BACKUP, enforce_eager=ENFORCE_EAGER)
smoke = config(PROJECT, RUNS, MODELS, phase='smoke', cap=128, **COMMON)
print(run(smoke))
smoke_rows = read_jsonl(phase_dir(smoke) / 'predictions.jsonl')
display(pd.DataFrame([{'condition': r['condition'], 'language': r['language'],
                      'tokens': r['tokens'], 'finish': r['finish_reason'],
                      'preview': r['output'][:160]} for r in smoke_rows]).head(14))

### 10. Check resume without rerunning completed work

**Why:** The output file is append-only, and each example has a stable key.

**Do:** Rerun the same smoke configuration. It should skip all completed keys without loading weights. This is a small recovery check, not a deliberate GPU interruption.

**Expected:** The number of saved records stays unchanged. Corrupt lines or conflicting duplicates cause an explicit error.

In [ ]:
before = len(read_jsonl(phase_dir(smoke) / 'predictions.jsonl'))
run(smoke)
after = len(read_jsonl(phase_dir(smoke) / 'predictions.jsonl'))
assert before == after
print('Resume check passed:', after, 'unique completed requests.')

### 11. Run the long-budget development pilot

**Why:** Measure real completion and runtime on this GPU. Short smoke outputs cannot predict the scientific result.

**Do:** Start with PILOT_CAP=16384 if time permits, otherwise 8192. Run A, B and R1 on the ten development pairs used in the recorded pilot. If 16384 is too slow, create a separate 8192 pilot before choosing that cap. Never use 4096 as the primary cap.

**Expected:** Real development timings and completion counts, saved separately from test data.

In [ ]:
PILOT_CAP = 8192  # Candidate: 8192 or 16384; select using available time, not test outcomes.
pilot = config(PROJECT, RUNS, MODELS, phase='pilot', cap=PILOT_CAP, arms=['A','B','R1'], **COMMON)
pilot['dev_pairs'] = 10
print(run(pilot))
pilot_summary, pilot_effects, pilot_coverage = analyze(pilot)
display(pilot_summary)
projections = pd.DataFrame(estimate(pilot))
display(projections)
fits = projections[(projections.conditions == 7) & projections.within_85min_with_reserve]
print('Largest 7-condition scope that fits the 85-minute window with 20% reserve:',
      int(fits.pairs.max()) if len(fits) else 'none: consider PILOT_CAP=8192, or a 5-condition pilot scope')
print('These are development results, not the held-out finding.')

### 12. Audit shorter-cap behavior on development prompts

**Why:** A saved prefix measures what happened along one long trajectory. Independently rerunning at a shorter cap may differ because scheduling can change sampling.

**Do:** Run the optional audit if the setup budget allows it. It uses one Swedish/English development pair for A/B at 1024 and 4096; inspect token-prefix agreement below.

**Expected:** A diagnostic table, not a guarantee of bitwise reproducibility. Differences do not invalidate the saved-trajectory definition.

In [ ]:
RUN_PREFIX_AUDIT = False  # Optional; set True before the test if time permits.
if RUN_PREFIX_AUDIT:
    comparisons = []
    long_rows = read_jsonl(phase_dir(pilot) / 'predictions.jsonl')
    for cap in [1024, 4096]:
        audit = config(PROJECT, RUNS, MODELS, phase='audit', cap=cap, arms=['A','B'],
                       **COMMON)
        audit['dev_pairs'] = 1
        run(audit)
        for short in read_jsonl(phase_dir(audit) / 'predictions.jsonl'):
            long = next(r for r in long_rows if (r['id'],r['language'],r['condition']) == (short['id'],short['language'],short['condition']))
            comparisons.append({'cap':cap,'condition':short['condition'],'language':short['language'],
                                'prefix_matches':short['token_ids'] == long['token_ids'][:len(short['token_ids'])]})
    display(pd.DataFrame(comparisons))
else:
    print('Audit omitted; report saved-trajectory curves without claiming independent-rerun equivalence.')

### 13. Freeze the test configuration

**Why:** Write down the choices before any test output. Prefer a useful long budget and fewer pairs over a cap that prevents measurement.

**Do:** Use the timing table to choose 40, 24 or 16 pairs. Keep all seven arms for the core. If necessary, explicitly label a five-arm run as a pilot and omit both random controls before testing. Explain the decision below. The recorded run used 40 pairs, all seven conditions and 8,192 tokens. DATA_REVIEWED must be True.

**Expected:** A frozen configuration with hashes and a runtime projection leaving 20% reserve. If nothing fits, stop and report the pilot instead of forcing a full run.

In [ ]:
N_PAIRS = 40   # Set from the 'Largest ... scope that fits' line printed by Section 11 (40, 24 or 16).
SCOPE = 'core'  # 'core' = 7 arms; 'pilot' = A, B, C, D, C_pen, explicitly limited.
conditions = ['A','C','C_pen','B','D','R1','S1'] if SCOPE == 'core' else ['A','C','C_pen','B','D']
projection = next(x for x in estimate(pilot) if x['pairs']==N_PAIRS and x['conditions']==len(conditions))
test = config(PROJECT, RUNS, MODELS, phase='test', cap=PILOT_CAP, pairs=N_PAIRS, arms=conditions,
              **COMMON)
test['data_reviewed'] = DATA_REVIEWED
test['scope'] = SCOPE
DECISION = 'Explain the pilot completion rates, timing, chosen cap and scope here.'
assert not DECISION.startswith('Explain'), 'Write the actual pre-test decision.'
test = freeze(test, DECISION, projection['projected_seconds'], pilot)
print('Frozen configuration:', phase_dir(test) / 'frozen.json')

### Run the frozen conditions, one group at a time

Each following cell opens a fresh GPU worker for one weight variant. Original weights supply A/C/C_pen; patched weights supply B/D. This avoids unverified changes inside a running engine. **Do not change settings after viewing these outputs.** If a worker fails, inspect its log, preserve all completed outputs, and rerun the same configuration only when the cause is resolved. A changed engine, dtype, code or dataset requires a separately declared run.

The 85-minute generation deadline persists across these group cells. An incomplete run is reported with missing requests. Do not reset the deadline to hide an overrun. Stop planned generation by the overall minute-170 target when that arrives first.

### 14. Run A, C and C_pen

**Why:** Collect the frozen A, C and C_pen comparison on the same selected examples.

**Do:** Run this cell, then wait for its completion message. Progress shows saved microbatches; full logs remain on disk.

**Expected:** Incremental predictions, optional Drive backup, and an MLflow condition record. Already completed keys are skipped.

In [ ]:
print(run(test, groups=['original']))

### 15. Run B and D

**Why:** Collect the frozen B and D comparison on the same selected examples.

**Do:** Run this cell, then wait for its completion message. Progress shows saved microbatches; full logs remain on disk.

**Expected:** Incremental predictions, optional Drive backup, and an MLflow condition record. Already completed keys are skipped.

In [ ]:
print(run(test, groups=['patched']))

### 16. Run R1

**Why:** Collect the frozen R1 comparison on the same selected examples.

**Do:** Run this cell, then wait for its completion message. Progress shows saved microbatches; full logs remain on disk.

**Expected:** Incremental predictions, optional Drive backup, and an MLflow condition record. Already completed keys are skipped.

In [ ]:
print(run(test, groups=['sign']))

### 17. Run S1

**Why:** Collect the frozen S1 comparison on the same selected examples.

**Do:** Run this cell, then wait for its completion message. Progress shows saved microbatches; full logs remain on disk.

**Expected:** Incremental predictions, optional Drive backup, and an MLflow condition record. Already completed keys are skipped.

In [ ]:
print(run(test, groups=['orthogonal']))

## Checks

Now analyze the saved outputs. The primary contrast is B−A on Swedish arithmetic. D−C on regression probes is the main regression screen; A/B probe failures can reflect looping or failure to finish. These small probes cannot establish that general capability is preserved.

### 18. Check coverage and compare outcomes

**Why:** A partial run must not look like a complete experiment. Paired comparisons use only matched IDs and disclose missingness.

**Do:** Read coverage first, then the summary and paired effects. Intervals resample wording families, keeping translations and numeric variants together.

**Expected:** Counts, matched transition tables, effect sizes and 95% intervals. Degenerate intervals are flagged; zero observed change does not establish equivalence.

In [ ]:
summary, effects, coverage = analyze(test)
print({k:v for k,v in coverage.items() if k!='missing'})
if coverage['missing']: print('Missing requests:', coverage['missing'][:10], '(full list in coverage.json)')
display(summary)
display(effects)

### 19. Read the figures

**Why:** Compare bounded success across conditions and see how it changes with output budget.

**Do:** Read the counts beside the bars. Budget curves come from prefixes of the same saved generations, not independent short runs or wall-clock measurements.

**Expected:** Two figures drawn only from actual outputs. Any partial coverage remains a limitation of these descriptive charts.

In [ ]:
from IPython.display import Image, display
folder = phase_dir(test)
display(Image(filename=str(folder / 'swedish_success.png')))
display(Image(filename=str(folder / 'budget_curves.png')))

### 20. Validate the language screen on a blind sample

**Why:** A stopword heuristic is cheap, but short, repetitive or mixed text can fool it. We do not assume it is reliable.

**Do:** Download language_blind_review.csv. Label up to 40 views as sv, en, mixed/other or insufficient without opening language_audit_key.csv. Upload the completed CSV in the next cell.

**Expected:** A manageable sample across the first 250 reasoning tokens, full reasoning, and final prose. Numeric-only answers are insufficient evidence of language.

In [ ]:
from google.colab import files
files.download(str(folder / 'language_blind_review.csv'))

### 21. Inspect detector disagreements

**Why:** Compare the blind manual labels with automatic labels before reporting language fidelity.

**Do:** Upload the labeled CSV. Do not tune detector thresholds on test labels; report the confusion table and limitations, or restrict language claims to the manually reviewed sample.

**Expected:** Sample agreement and a class-by-class confusion table. There is no universal accuracy guarantee.

In [ ]:
uploaded_labels = files.upload()
assert len(uploaded_labels) == 1
label_bytes = next(iter(uploaded_labels.values()))
(folder / 'language_blind_review.csv').write_bytes(label_bytes)
validation, confusion = validate_language_audit(folder)
print(validation)
display(confusion)

### 22. Generate the research-note draft

**Why:** Turn recorded evidence into a short report without inventing a conclusion.

**Do:** Generate the Markdown/PDF draft, then write your recommendation and review the PDF before submission. The draft is explicitly labeled incomplete until that author step is done.

**Expected:** A research-note draft containing the observed primary result, coverage and limitations, plus a clearly marked recommendation section to complete.

In [ ]:
note = write_report(test)
print(note.read_text(encoding='utf-8'))
print('Review the PDF and complete the recommendation before submission.')

### 23. Save the complete evidence package

**Why:** Keep portable outputs even if MLflow or Colab later becomes unavailable.

**Do:** Run this cell before disconnecting. Download the ZIP, save a copy of this executed notebook, and update AI_USAGE.md and the effort log. Review contents before any result publication.

**Expected:** A ZIP with predictions, configuration, manifests, environment lock, summaries, figures and draft report. Model binaries and credentials are excluded.

In [ ]:
import shutil
from sellma_wsp.common import checkpoint, snapshot_tracking
snapshot_tracking(folder, folder / 'tracking_snapshot')
shutil.copy2(MODELS / 'weights_manifest.json', folder / 'weights_manifest.json')
archive = shutil.make_archive('/content/sellma_results', 'zip', RUNS)
if BACKUP:
    checkpoint(archive, BACKUP / 'sellma_results.zip')
files.download(archive)
print('Session elapsed minutes:', round((time.time()-SESSION_START)/60,1))
print('Add prior preparation time to the effort ledger; this is not total project effort.')

## Next Steps

1. Write the recommendation using B−A, B−C, B−C_pen, D−C and control results together. A nonsignificant difference does not establish equivalence.
2. Include representative successes and failures, with condition names restored only after blind review.
3. Check the 1–2-page Markdown note and PDF, code revision, seeds, outputs and AI-use disclosure against the assignment.
4. Commit compact results and your executed notebook to the private repository. Do not commit weights, caches, credentials or a live tracking database.
5. Decide on public visibility for final submission separately; the assignment requests a public repository.

**Troubleshooting:** no GPU → change runtime; BF16 error → prefer L4/A100 or declare an FP16 pilot; package/kernel error → inspect worker log and use one verified engine for every arm; slow generation → reduce pairs before shortening below 8,192; corrupt JSONL → preserve it and repair explicitly; missing results → report partial coverage. Do not silently replace the model, drop inconvenient conditions, or select the best of retries.

**Sources:** [published experiment](https://github.com/RikiyaT/lm_understand-logs/tree/4d09ba6d6a7e891dd8063974dfe81b3d78e3494f/exp8_math), [Qwen3.5 documentation](https://huggingface.co/docs/transformers/model_doc/qwen3_5), [vLLM engine configuration](https://docs.vllm.ai/en/stable/configuration/engine_args/). Source licenses are retained in `vendor/upstream/LICENSE` and `THIRD_PARTY_NOTICES.md`.